# Musanga V4 — NB04 V4.006: safe MS metadata and limited native-window QA

The original MS data have full-width, **single-row GeoTIFF strips** (except tiled Baego), so raster-wide previews and random scatter samples can cause excessive memory use. **Do not use WarpedVRT or raster-wide downsampling.**

Cells 00–03 read metadata only. **New cell 04** inspects at most three small *64×64 native-resolution windows* per MS raster, one file at a time, with GDAL caching capped. It measures both bands separately (including possible transparency/alpha), without assuming radiometric calibration.

No U-Net training, no raw datasets modified, no full mosaic in RAM. All new outputs write to `Outputs/v4/04_ms_qa_v4_004`. Native-window results are samples only, not full ROI quality metrics.

**New cell 05:** alpha-aware diagnosis of MOS 5–6 red data and byte-for-byte *sampled pixel* comparison of MOS 1–2 red-edge duplicates. It uses a bounded 6×6 geographic grid and 16×16 native windows; it never reads a full raster. The previous cell 04's `mean_valid` ignores alpha and must not be treated as genuine MS validity.

**New cells 06–08:** paired sampled MS/RGB classifier using read-only NB02 V4.003 labelled chips and NB03 recorded spatial inner roles. MOS 5–6 is excluded entirely. No U-Net training, no full mosaics, strict alpha-valid native 16×16 reads. Experimental results are *screening only*, pending registration and calibration QA.


In [ ]:
# 00 — Reuse NB00 inventory and frozen ROI; metadata only, no image read
from pathlib import Path
import json, math, re, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.warp import transform_bounds
from rasterio.vrt import WarpedVRT
from rasterio.transform import from_bounds
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
DATA=ROOT/'Data'
NB00=ROOT/'Outputs/v4/00_input_audit'
OUT=ROOT/'Outputs/v4/04_ms_qa_v4_006'
OUT.mkdir(parents=True,exist_ok=True)
REG=NB00/'00_ms_registry.json'
HEADERS=NB00/'00_ms_raster_headers.csv'
OVERLAP=NB00/'00_ms_rgb_extent_audit.csv'
assert REG.is_file() and HEADERS.is_file() and OVERLAP.is_file(),(
    'First run NB00 MS inventory cells; missing',REG,HEADERS,OVERLAP)
ms_manifest=json.loads(REG.read_text())
ms=pd.read_csv(HEADERS).fillna({'error':''})
overlap=pd.read_csv(OVERLAP)
ROI_PATH=DATA/'Ortho_RGB'/'ROI.shp'
roi=gpd.read_file(ROI_PATH)
assert roi.crs is not None and len(roi)>0
RGB_REFERENCE=ms_manifest['rgb_reference']
RGB_DIR=DATA/'Ortho_RGB'
RGB_FILES={group:RGB_DIR/name for group,name in RGB_REFERENCE.items()}
assert all(p.is_file() for p in RGB_FILES.values()),RGB_FILES
EDGE_DISTANCE_M=(1,3,5,10,20,30,50)
print('Available surveys:', sorted(ms.group.dropna().unique()))
print('NB00 MS records:',len(ms),'| paired extent records:',len(overlap))
display(ms[['group','band_hint','name','bands','dtype','nodata','error']].head(40))


In [ ]:
# 01 — Band headers only, ZERO source pixel reads
rows=[]
for rec in ms.itertuples():
    if getattr(rec,'error',''):continue
    path=Path(rec.path)
    if not path.is_file():continue
    with rasterio.Env(GDAL_CACHEMAX=32):
        with rasterio.open(path) as ds:
            for b in range(1,ds.count+1):
                rows.append(dict(group=rec.group,file=path.name,path=str(path),
                    band=b,band_hint=rec.band_hint,
                    description=ds.descriptions[b-1],
                    colorinterp=str(ds.colorinterp[b-1]),
                    dtype=ds.dtypes[b-1],scale=ds.scales[b-1],offset=ds.offsets[b-1],
                    nodata=ds.nodatavals[b-1],crs=str(ds.crs),
                    width=ds.width,height=ds.height,
                    pixel_x_m=abs(ds.transform.a),pixel_y_m=abs(ds.transform.e),
                    block_shape=str(ds.block_shapes[b-1]),
                    pixel_distribution='NOT SAMPLED',file_bytes=path.stat().st_size))
    print('HEADERS:',rec.group,rec.name,flush=True)
BAND_AUDIT=pd.DataFrame(rows)
assert not BAND_AUDIT.empty
BAND_AUDIT.to_csv(OUT/'04_ms_band_metadata_only.csv',index=False)
display(BAND_AUDIT[['group','file','band','band_hint','description','dtype',
                    'scale','offset','nodata','pixel_x_m','block_shape']])
print('No pixel data read, no band-value statistics inferred.')


In [ ]:
# 02 — ROI / RGB / MS BOUNDING-BOX overlap ONLY, ZERO pixel reads
from shapely.geometry import box
from shapely.ops import unary_union
from rasterio.warp import transform_bounds
summary=[]
for group,records in ms.groupby('group',sort=True):
    if group not in RGB_FILES:continue
    with rasterio.open(RGB_FILES[group]) as rgb:
        study=unary_union(roi.to_crs(rgb.crs).geometry).intersection(box(*rgb.bounds))
        area=float(study.area)
        if area<=0:continue
        for rec in records.itertuples():
            if getattr(rec,'error',''):continue
            path=Path(rec.path)
            if not path.is_file():continue
            with rasterio.Env(GDAL_CACHEMAX=32):
                with rasterio.open(path) as src:
                    if src.crs is None:continue
                    bb=box(*transform_bounds(src.crs,rgb.crs,*src.bounds,densify_pts=21))
                    frac=float(study.intersection(bb).area/area)
                    summary.append(dict(group=group,file=rec.name,band_hint=rec.band_hint,
                        nominal_bands=int(src.count),roi_bbox_overlap_fraction=frac,
                        rgb_roi_area_ha=area/1e4,valid_pixel_coverage='NOT TESTED',
                        registration='NOT TESTED'))
            print('BBOX:',group,rec.name,round(frac,3),flush=True)
SUMMARY=pd.DataFrame(summary)
assert not SUMMARY.empty
SUMMARY.to_csv(OUT/'04_ms_rgb_roi_bbox_only.csv',index=False)
display(SUMMARY.sort_values(['group','file']).round(3))
print('Bounding-box overlap is only an upper limit on valid MS pixel coverage.')


In [ ]:
# 03 — Metadata coverage figure, NO pixel reads
d=SUMMARY.sort_values(['group','file'])
fig,ax=plt.subplots(figsize=(9,max(4,len(d)*.23)))
ax.barh(np.arange(len(d)),d.roi_bbox_overlap_fraction.to_numpy())
ax.set_yticks(np.arange(len(d)))
ax.set_yticklabels([f"{r.group}: {r.file[:40]}" for r in d.itertuples()],fontsize=7)
ax.invert_yaxis()
ax.set_xlim(0,1.02)
ax.set_xlabel('Bounding-box overlap with RGB ∩ ROI (not actual valid coverage)')
ax.set_title('MS inventory — metadata only')
fig.tight_layout()
fig.savefig(OUT/'04_ms_bbox_metadata_summary.png',dpi=120,bbox_inches='tight')
plt.show()
plt.close(fig)
print('DONE:',OUT)


In [ ]:
# 04 — STRICTLY BOUNDED native-window MS sampling; do not resample whole images.
# 25 files x at most 3 windows x <= 2/4 bands x 64x64 pixels.
# Full-width strip TIFFs may decode an entire scanline internally; never request
# hundreds of rows, use WarpedVRT, or keep open handles/arrays between files.
from rasterio.warp import transform as transform_coords
from rasterio.windows import Window
from shapely.geometry import Point,box
from shapely.ops import unary_union
import gc

WINDOW_SIZE=64
MAX_PATCHES_PER_FILE=3
QA_ROWS=[]
QA_PATCHES=[]
QA_WARNINGS=[]
for group,records in ms.groupby('group',sort=True):
    if group not in RGB_FILES:continue
    with rasterio.open(RGB_FILES[group]) as rgb:
        crs=rgb.crs
        study=unary_union(roi.to_crs(crs).geometry).intersection(box(*rgb.bounds))
        if study.is_empty:continue
        for rec in records.itertuples():
            if getattr(rec,'error',''):continue
            path=Path(rec.path)
            if not path.is_file():continue
            print('NATIVE WINDOWS:',group,path.name,flush=True)
            # Only one source open; cap GDAL's decompressed block cache at 16 MB.
            with rasterio.Env(GDAL_CACHEMAX=16):
                with rasterio.open(path) as src:
                    if src.crs is None:continue
                    from rasterio.warp import transform_bounds
                    source_box=box(*transform_bounds(src.crs,crs,*src.bounds,densify_pts=21))
                    relevant=study.intersection(source_box)
                    if relevant.is_empty:continue
                    bb=relevant.bounds
                    choices=[relevant.representative_point()]
                    # Deterministic geographic candidates, no raster samples/read.
                    for u,v in [(0.25,0.25),(0.75,0.75),(0.25,0.75),
                                (0.75,0.25),(0.5,0.5),(0.35,0.65),(0.65,0.35)]:
                        pt=Point(bb[0]+u*(bb[2]-bb[0]),bb[1]+v*(bb[3]-bb[1]))
                        if relevant.covers(pt):choices.append(pt)
                    seen=set()
                    count=0
                    for pt in choices:
                        if count>=MAX_PATCHES_PER_FILE:break
                        if src.crs==crs:x,y=pt.x,pt.y
                        else:
                            mx,my=transform_coords(crs,src.crs,[pt.x],[pt.y])
                            x,y=mx[0],my[0]
                        row,col=src.index(x,y)
                        if row<0 or col<0 or row>=src.height or col>=src.width:continue
                        col=int(np.clip(col-WINDOW_SIZE//2,0,max(0,src.width-WINDOW_SIZE)))
                        row=int(np.clip(row-WINDOW_SIZE//2,0,max(0,src.height-WINDOW_SIZE)))
                        key=(row//WINDOW_SIZE,col//WINDOW_SIZE)
                        if key in seen:continue
                        seen.add(key)
                        win=Window(col,row,min(WINDOW_SIZE,src.width-col),
                                   min(WINDOW_SIZE,src.height-row))
                        # Never read all bands at once; no large intermediate arrays.
                        pixel_arrays=[]
                        for band in range(1,src.count+1):
                            a=src.read(band,window=win,masked=True)
                            v=np.asarray(a.data)
                            valid=(~np.ma.getmaskarray(a))&np.isfinite(v)
                            vals=v[valid]
                            unique=np.unique(v) if v.size<=4096 else np.array([])
                            QA_ROWS.append(dict(group=group,file=path.name,
                                band_hint=rec.band_hint,band=band,patch=count,
                                row=row,col=col,n_window=int(v.size),
                                n_valid=int(valid.sum()),
                                valid_fraction=float(valid.mean()),
                                min=float(vals.min()) if len(vals) else np.nan,
                                p05=float(np.percentile(vals,5)) if len(vals) else np.nan,
                                median=float(np.median(vals)) if len(vals) else np.nan,
                                p95=float(np.percentile(vals,95)) if len(vals) else np.nan,
                                max=float(vals.max()) if len(vals) else np.nan,
                                zero_frac_valid=float(np.mean(vals==0)) if len(vals) else np.nan,
                                fraction_65535=float(np.mean(vals==65535)) if len(vals) else np.nan,
                                n_unique=int(len(unique)),
                                unique_values=str(unique[:12].tolist()) if len(unique)<=12 else 'many'))
                            # Caution: 2-band TIFF may be intensity + alpha. Report, do
                            # not pre-assign alpha without a diagnostic.
                            del a,v,valid,vals
                        count+=1
                    if count==0:QA_WARNINGS.append((group,path.name,'No in-footprint native patch'))
            print('  COMPLETED:',count,'native 64x64 patches; GDAL cache capped',flush=True)
            gc.collect()

PATCH_QA=pd.DataFrame(QA_ROWS)
assert not PATCH_QA.empty,'No valid sample windows in any MS raster'
PATCH_QA.to_csv(OUT/'04_native_patch_band_statistics.csv',index=False)
summ=(PATCH_QA.groupby(['group','file','band_hint','band'],dropna=False)
      .agg(n_patches=('patch','nunique'),mean_valid=('valid_fraction','mean'),
           median_p05=('p05','median'),median=('median','median'),
           median_p95=('p95','median'),mean_zero=('zero_frac_valid','mean'),
           mean_65535=('fraction_65535','mean'),
           median_n_unique=('n_unique','median')).reset_index())
summ.to_csv(OUT/'04_native_patch_summary.csv',index=False)
display(summ.round(3))
if QA_WARNINGS:print('NO VALID WINDOWS:',QA_WARNINGS)
print('Sample distributions are NOT proof of calibrated reflectance or registration.')
print('Native-window audit complete:',OUT)


In [ ]:
# 05 — Alpha-aware paired spectral QA (MOS 5–6 red and MOS 1–2 red-edge duplicates)
# Read only 16x16 source windows, at most 36 coordinates per survey.
# A zero in band 1 with alpha == 0 is TRANSPARENT, not evidence of corruption.
from shapely.geometry import Point
from rasterio.warp import transform as warp_coords
from rasterio.windows import Window
from itertools import combinations
import gc
SAMPLE_N=6
SIZE=16
qa_rows=[]
groups_to_check=('MOS_5_6_MS','MOS_1_2_MS')
for group in groups_to_check:
    recs=ms[(ms.group==group) & (ms.band_hint.isin(['red','green','nir','red_edge']))].copy()
    assert not recs.empty
    rgb_path=RGB_FILES[group]
    with rasterio.open(rgb_path) as rgb:
        base_crs=rgb.crs
        study=unary_union(roi.to_crs(base_crs).geometry).intersection(box(*rgb.bounds))
        # Limit to MS union intersection: prevents wasting all samples outside imagery.
        boxes=[]
        for rec in recs.itertuples():
            with rasterio.open(rec.path) as src:
                boxes.append(box(*transform_bounds(src.crs,base_crs,*src.bounds,densify_pts=21)))
        footprint=study.intersection(unary_union(boxes))
        assert not footprint.is_empty,group
        xmin,ymin,xmax,ymax=footprint.bounds
        test_points=[]
        for fy in np.linspace(.08,.92,SAMPLE_N):
            for fx in np.linspace(.08,.92,SAMPLE_N):
                pt=Point(xmin+fx*(xmax-xmin),ymin+fy*(ymax-ymin))
                if footprint.covers(pt):test_points.append(pt)
        if not test_points:
            test_points=[footprint.representative_point()]
    print('ALPHA QA:',group,'candidate shared locations',len(test_points),flush=True)
    # Open only a single raster at a time. Save tiny arrays for cross-band QA.
    sampled={}
    for rec in recs.itertuples():
        collected={}
        with rasterio.Env(GDAL_CACHEMAX=8):
            with rasterio.open(rec.path) as src:
                for j,pt in enumerate(test_points):
                    if src.crs==base_crs: xx,yy=pt.x,pt.y
                    else:
                        x2,y2=warp_coords(base_crs,src.crs,[pt.x],[pt.y])
                        xx,yy=x2[0],y2[0]
                    row,col=src.index(xx,yy)
                    if row<SIZE or col<SIZE or row>=src.height-SIZE or col>=src.width-SIZE:continue
                    win=Window(col-SIZE//2,row-SIZE//2,SIZE,SIZE)
                    # Read band 1 and (when present) alpha-like band 2 only.
                    v=src.read(1,window=win,masked=True)
                    z=np.asarray(v.data)
                    valid=(~np.ma.getmaskarray(v)) & np.isfinite(z)
                    alpha=None
                    if src.count>=2:
                        a=src.read(2,window=win,masked=False)
                        alpha=np.asarray(a)
                        # Explicitly treat alpha=0 as invisible. Non-zero may include
                        # partial transparency; record it separately.
                        valid &= alpha>0
                    values=z[valid]
                    qa_rows.append(dict(group=group,file=rec.name,band=rec.band_hint,
                        location=j,valid_fraction=float(valid.mean()),
                        valid_pixels=int(valid.sum()),
                        alpha_zero_fraction=float(np.mean(alpha==0)) if alpha is not None else np.nan,
                        alpha_full_fraction=float(np.mean(alpha==255)) if alpha is not None else np.nan,
                        signal_median=float(np.median(values)) if len(values) else np.nan,
                        signal_zero_fraction=float(np.mean(values==0)) if len(values) else np.nan))
                    collected[j]=(z.copy(),valid.copy())
        sampled[rec.name]=collected
        print('  ',rec.name,'windows',len(collected),'valid locations',
              sum(mask.sum()>0 for z,mask in collected.values()),flush=True)
        gc.collect()
    if group=='MOS_1_2_MS':
        rededges=[p for p in recs[recs.band_hint.eq('red_edge')].name]
        if len(rededges)==2:
            a,b=(sampled[x] for x in rededges)
            shared=sorted(set(a)&set(b))
            different=0;valid_pixels=0
            for k in shared:
                za,ma=a[k];zb,mb=b[k]
                joint=ma&mb
                different+=int(np.count_nonzero(za[joint]!=zb[joint]))
                valid_pixels+=int(joint.sum())
            print('DUPLICATE RED EDGE: differing shared valid sampled pixels:',
                  different,'/',valid_pixels,'across',len(shared),'windows')
            if valid_pixels==0:
                print('  INCONCLUSIVE: no shared valid pixels sampled.')
            elif different==0:
                print('  Sampled values identical; full-file identity NOT proven.')
    del sampled
qa=pd.DataFrame(qa_rows)
qa.to_csv(OUT/'04_alpha_aware_paired_QA.csv',index=False)
aggregate=(qa.groupby(['group','band','file'],as_index=False).agg(
    n_windows=('location','nunique'),
    windows_with_valid=('valid_pixels',lambda v:int((v>0).sum())),
    mean_valid_fraction=('valid_fraction','mean'),
    mean_alpha_zero_fraction=('alpha_zero_fraction','mean'),
    median_signal=('signal_median','median')))
display(aggregate.round(3))
print('QA saved:',OUT/'04_alpha_aware_paired_QA.csv')
print('NOTE: these are sparse native windows, not an estimate of full-area valid coverage.')


## Interpretation after cell 05

The previous cell 04 can report mean_valid=1 even when both spectral values and alpha are zero. Cell 05 explicitly checks the second channel as **candidate alpha**: pixels with zero in channel 2 are rejected. A high fraction of zero-alpha windows for MOS 5–6 red would indicate missing coverage, **not necessarily radiometric corruption**.

When valid red pixels exist, compare their paired local signals with green, NIR and red-edge, and then inspect RGB/MS alignment before computing spectral indices. MOS 1–2 duplicate red-edge filenames are tested at identical native locations; matching samples are not proof of full-file identity. No original dataset is modified.


## 06 — First supervised spectral discrimination (screening experiment)

**Goal:** compare *RGB pixel values* against *RGB + MS* at the exact same annotated pixel locations, using the existing strictly spatially separated **inner training/validation roles** recorded by NB03. This is a small, efficient screening model, **not** a replacement for RGB U-Net, and NOT an outer held-out evaluation.

**Exclusions:** MOS 5–6 entire survey (known red-alpha failure), Yoko (no MS), 4-band group1 MOS RGBA browse imagery (not MS). Yangambi YBForgeo/YBJoseph stay **separate acquisitions**; samples may be unavailable if annotated chips do not intersect.

**Critical safeguards:** use both bands of uint16 MS TIFFs (band1 signal, band2 alpha); no 0-alpha pixels; 16×16 native windows only; no mosaics warped; spectral indices are NOT interpreted as reflectance indices. No MS-to-RGB subpixel registration verification, so results exploratory only. Labels may be partial in original chips; **we use only exhaustive labels** (source codes 2/3) for both train and validation.

**Per-fold sampling:** up to 30 training and 12 selection chips per acquisition, 2 positive + 2 negative labelled pixels per chip, for bounded I/O. Use NB03's audited exact inner split; fail closed if it is missing. Avoid any sample near another role's annotation through that audited split.


In [ ]:
# 06A — Read existing labels + strict inner split AUDIT (no new rasterization)
from scipy.special import expit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from rasterio.windows import Window
from rasterio.transform import xy as pixel_xy
from rasterio.warp import transform as warp_xy
from collections import defaultdict
import gc, os, hashlib
from contextlib import ExitStack

FOLDS=(0,1,2)
CHIP_SIZE=256
LABEL_DIR=ROOT/'Outputs/v4/02_rgb_unet_field78_v4_003_full/chip_cache'
SPLIT_DIR=ROOT/'Outputs/v4/03_rgb_dsm_multiscale_v4_007'
assert LABEL_DIR.is_dir() and SPLIT_DIR.is_dir()
PER_ACQUISITION_TRAIN_CHIPS=30
PER_ACQUISITION_VAL_CHIPS=12
PIXELS_PER_CLASS_PER_CHIP=2
PATCH_SIDE=16
SEED_MS=20261116
MS_EXP=OUT/'06_spatial_ms_screen'
MS_EXP.mkdir(parents=True,exist_ok=True)
MS_BANDS=('red','green','red_edge','nir')
ALLOWED_GROUPS=('Baego_MS','MOS_1_2_MS','MOS_3_4_MS',
                'Yangambi_YBJoseph_right_MS','Yangambi_YBForgeo_right_MS')
assert 'MOS_5_6_MS' not in ALLOWED_GROUPS

def registry_build():
    regs={}
    for grp in ALLOWED_GROUPS:
        part=ms[ms.group.eq(grp)]
        if part.empty:continue
        bands={}
        for b in MS_BANDS:
            cand=part[part.band_hint.eq(b)].copy()
            if grp=='Baego_MS':
                assert len(part)==1 and int(part.iloc[0].bands)==4
                # Explicitly certified band descriptions from 00 metadata audit.
                with rasterio.open(part.iloc[0].path) as ds:
                    desc=[str(s).lower().replace(' ','') for s in ds.descriptions]
                assert desc==['red','green','rededge','nir'],desc
                bands[b]=(str(part.iloc[0].path),MS_BANDS.index(b)+1,None)
            else:
                if b=='red_edge' and grp=='MOS_1_2_MS':
                    # Two sampled-identical red-edge files. Select the unsuffixed one.
                    cand=cand[cand.name.eq('mlt_Mos_1_2_transparent_mosaic_red edge_georef.tif')]
                if len(cand)!=1:
                    raise RuntimeError(f'Missing or ambiguous MS band: {grp} {b}: {list(cand.name)}')
                bands[b]=(str(cand.iloc[0].path),1,2)
        regs[grp]=bands
    return regs

MS_REG=registry_build()
print('Selected MS acquisitions:',list(MS_REG))
print('EXCLUDED: MOS 5–6 (all spectral files); Yoko (no MS).')
print('All comparisons use identical sampled pixels for RGB vs RGB+MS.')

def labelled_chip_cache(fold):
    paths=sorted(LABEL_DIR.glob(f'field78_train_fold{fold}_*.npz'))
    assert len(paths)==1,('Ambiguous or missing frozen source-labelled cache',fold,paths)
    with np.load(paths[0],allow_pickle=False) as z:
        # Each compressed NPZ member is decompressed ONCE; prevents RAM spikes.
        arrays={k:z[k] for k in ('rgb','y','w','source','site','mosaic','row','col')}
        fingerprint=str(z['key'].item())
    n=len(arrays['rgb'])
    assert arrays['rgb'].shape==(n,CHIP_SIZE,CHIP_SIZE,3)
    assert arrays['y'].shape==arrays['w'].shape==arrays['source'].shape==(n,CHIP_SIZE,CHIP_SIZE)
    return arrays,fingerprint

def recorded_inner_roles(fold):
    candidates=[SPLIT_DIR/f'03_v4_002_inner_spatial_split_fold{fold}.csv']
    candidates += [ROOT/'Outputs/v4/03_rgb_dsm_field78_v4_002'/f'03_v4_002_inner_spatial_split_fold{fold}.csv']
    paths=[p for p in candidates if p.is_file()]
    assert paths,('Missing NB03 recorded spatial split; do NOT invent new split',fold)
    d=pd.read_csv(paths[0])
    assert {'mosaic','row','col','inner_role','site'}.issubset(d.columns),d.columns.tolist()
    assert {'inner_train','inner_val_candidate'}.issubset(set(d.inner_role))
    assert not d.duplicated(['mosaic','row','col']).any()
    return d,paths[0]

def rgb_transforms():
    keys=set(ms_manifest['rgb_reference'].values())
    transforms={}
    for name in keys:
        p=RGB_DIR/name
        if p.is_file():
            with rasterio.open(p) as ds:transforms[name]=(ds.transform,ds.crs)
    return transforms
RGB_GEO=rgb_transforms()

def candidate_acquisitions(mosaic):
    key=str(mosaic).lower()
    if 'baego' in key:return ['Baego_MS']
    if 'mos_1_2' in key:return ['MOS_1_2_MS']
    if 'mos_3_4' in key:return ['MOS_3_4_MS']
    if 'mos_5_6' in key:return []  # explicit whole-site EXCLUSION
    if 'yangambi' in key:return ['Yangambi_YBJoseph_right_MS','Yangambi_YBForgeo_right_MS']
    return []

def select_samples(fold):
    a,sig=labelled_chip_cache(fold)
    role_df,path=recorded_inner_roles(fold)
    role_map={(str(q.mosaic),int(q.row),int(q.col)):str(q.inner_role)
              for q in role_df.itertuples(index=False)}
    rng=np.random.default_rng(SEED_MS+fold)
    candidates=defaultdict(list)
    counts=defaultdict(int)
    for i in range(len(a['rgb'])):
        mosaic=str(a['mosaic'][i]);site=str(a['site'][i])
        role=role_map.get((mosaic,int(a['row'][i]),int(a['col'][i])))
        if role not in ('inner_train','inner_val_candidate'):continue
        if mosaic not in RGB_GEO:continue
        source=a['source'][i]
        labelled=a['w'][i].astype(bool)&np.isin(source,(2,3))
        y=a['y'][i].astype(bool)
        pos=np.flatnonzero((labelled&y).ravel())
        neg=np.flatnonzero((labelled&~y).ravel())
        if len(pos)<2 or len(neg)<2:continue
        for group in candidate_acquisitions(mosaic):
            candidates[group,role].append(i)
    rows=[]
    for (group,role),ids in sorted(candidates.items()):
        maximum=PER_ACQUISITION_TRAIN_CHIPS if role=='inner_train' else PER_ACQUISITION_VAL_CHIPS
        chosen=rng.choice(ids,size=min(maximum,len(ids)),replace=False)
        for i in chosen:
            mosaic=str(a['mosaic'][i])
            t,crs=RGB_GEO[mosaic]
            source=a['source'][i]
            labelled=a['w'][i].astype(bool)&np.isin(source,(2,3))
            truth=a['y'][i].astype(bool)
            pix=[]
            for mask in (labelled&truth,labelled&~truth):
                idx=np.flatnonzero(mask.ravel())
                pix.extend(rng.choice(idx,size=min(PIXELS_PER_CLASS_PER_CHIP,len(idx)),
                                      replace=False))
            for index in pix:
                rr,cc=np.unravel_index(int(index),(CHIP_SIZE,CHIP_SIZE))
                ypix=int(a['row'][i])+int(rr);xpix=int(a['col'][i])+int(cc)
                x,y_coord=pixel_xy(t,ypix,xpix,offset='center')
                rgb=a['rgb'][i,rr,cc].astype(np.float32)/255.
                rows.append(dict(fold=fold,acquisition=group,site=str(a['site'][i]),
                    role=role,mosaic=mosaic,chip=f'{mosaic}|{int(a["row"][i])}|{int(a["col"][i])}',
                    x=float(x),ycoord=float(y_coord),crs=str(crs),label=int(truth[rr,cc]),
                    R=float(rgb[0]),G=float(rgb[1]),B=float(rgb[2])))
    del a
    gc.collect()
    frame=pd.DataFrame(rows)
    if frame.empty:raise RuntimeError(f'No eligible labelled pixels in fold {fold}')
    assert frame.groupby(['acquisition','role']).size().max()<=4*PER_ACQUISITION_TRAIN_CHIPS
    print('FOLD',fold,'read-only audit:',path.name,'source key',sig[:12],
          '| candidate pixels',len(frame),flush=True)
    return frame


In [ ]:
# 06B — Source-native 16x16 patch medians, alpha-masked. No WarpedVRT.
# One file at a time, reopen each source once per group; calculate independent
# coordinate window reads capped at 16 x 16. No index computations on uncalibrated DN.
def read_ms_pixels(frame):
    out=frame.copy()
    for band in MS_BANDS:
        out[f'MS_{band}']=np.nan
        out[f'MS_{band}_valid_fraction']=0.
    for group,g in frame.groupby('acquisition',sort=True):
        print('SAMPLING',group,'candidate pixels',len(g),flush=True)
        for band in MS_BANDS:
            path,bidx,aidx=MS_REG[group][band]
            vals=[];valids=[]
            with rasterio.Env(GDAL_CACHEMAX=8):
                with rasterio.open(path) as ds:
                    # Assume all chip RGB CRS share the survey's target CRS.
                    coords=g[['x','ycoord']].to_numpy(float)
                    src_crs=g.crs.iloc[0]
                    if ds.crs!=rasterio.crs.CRS.from_string(src_crs):
                        mx,my=warp_xy(src_crs,ds.crs,coords[:,0].tolist(),coords[:,1].tolist())
                        coords=np.column_stack([mx,my])
                    for x,y in coords:
                        rr,cc=ds.index(float(x),float(y))
                        if rr<8 or cc<8 or rr>=ds.height-8 or cc>=ds.width-8:
                            vals.append(np.nan);valids.append(0.);continue
                        win=Window(cc-8,rr-8,PATCH_SIDE,PATCH_SIDE)
                        patch=ds.read(bidx,window=win,masked=True)
                        z=np.asarray(patch.data,dtype=np.float32)
                        v=~np.ma.getmaskarray(patch)&np.isfinite(z)
                        if aidx is not None:
                            alpha=ds.read(aidx,window=win,masked=False)
                            v&=(alpha>0)
                        # Avoid boundary/transparent patches: 90% valid signal.
                        frac=float(v.mean())
                        sig=float(np.median(z[v])) if frac>=.90 else np.nan
                        vals.append(sig);valids.append(frac)
                        del patch
            out.loc[g.index,f'MS_{band}']=vals
            out.loc[g.index,f'MS_{band}_valid_fraction']=valids
        good=out.loc[g.index,[f'MS_{b}' for b in MS_BANDS]].notna().all(axis=1)
        print(' ',group,'complete four-band samples',int(good.sum()),'/',len(g),flush=True)
        gc.collect()
    return out

samples=[]
for fold in FOLDS:
    cache=MS_EXP/f'04_ms_paired_fold{fold}.csv'
    if cache.is_file():
        q=pd.read_csv(cache)
        assert {'fold','role','acquisition','chip','MS_red','MS_nir','label'}.issubset(q.columns)
        assert set(q.fold)=={fold}
        print('READ CACHED',cache.name,len(q),flush=True)
    else:
        candidate=select_samples(fold)
        q=read_ms_pixels(candidate)
        tmp=cache.with_suffix('.tmp')
        q.to_csv(tmp,index=False);os.replace(tmp,cache)
        print('SAVED',cache.name,flush=True)
    samples.append(q)
ALL_SAMPLES=pd.concat(samples,ignore_index=True)
ALL_SAMPLES['ms_complete']=ALL_SAMPLES[[f'MS_{b}' for b in MS_BANDS]].notna().all(axis=1)
display(ALL_SAMPLES.groupby(['fold','acquisition','role']).agg(
    n=('label','size'),complete=('ms_complete','sum'),
    positive=('label','sum')).reset_index())
print('All candidate sample rows saved; original rasters and label caches untouched.')


In [ ]:
# 07 — Matched-support RGB vs RGB+MS evaluation (CPU only)
# Note: MS values are uncalibrated DN. Train and validate within the SAME
# acquisition, never cross-flight radiometric extrapolation.
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
PREDICTORS_RGB=['R','G','B']
PREDICTORS_MS=[f'MS_{b}' for b in MS_BANDS]
rows=[]
for fold in FOLDS:
    frame=ALL_SAMPLES[(ALL_SAMPLES.fold==fold)&ALL_SAMPLES.ms_complete].copy()
    for group in ALLOWED_GROUPS:
        q=frame[frame.acquisition==group].copy()
        tr=q[q.role.eq('inner_train')]
        va=q[q.role.eq('inner_val_candidate')]
        # AUC requires meaningful held-out positives and negatives, and at
        # least 3 independent training/selection chips in each group.
        if (len(tr)<35 or len(va)<12 or tr.chip.nunique()<5 or
            va.chip.nunique()<3 or tr.label.nunique()<2 or va.label.nunique()<2):
            rows.append(dict(fold=fold,acquisition=group,status='insufficient matched labels',
                             n_train=len(tr),n_val=len(va),
                             n_train_chips=tr.chip.nunique(),n_val_chips=va.chip.nunique(),
                             auc_rgb=np.nan,auc_rgb_ms=np.nan,delta_auc=np.nan))
            print('SKIP',fold,group,'insufficient labels/coverage',flush=True)
            continue
        assert set(tr.chip).isdisjoint(set(va.chip)),('Leakage!',fold,group)
        scores={}
        for name,cols in [('RGB',PREDICTORS_RGB),('RGB_MS',PREDICTORS_RGB+PREDICTORS_MS)]:
            model=HistGradientBoostingClassifier(max_iter=70,max_depth=3,
                        max_leaf_nodes=8,learning_rate=.06,l2_regularization=3,
                        early_stopping=False,random_state=SEED_MS)
            model.fit(tr[cols].to_numpy(np.float32),tr.label.to_numpy())
            scores[name]=roc_auc_score(va.label,model.predict_proba(
                va[cols].to_numpy(np.float32))[:,1])
        result=dict(fold=fold,acquisition=group,status='evaluated',
            n_train=len(tr),n_val=len(va),n_train_chips=tr.chip.nunique(),
            n_val_chips=va.chip.nunique(),auc_rgb=scores['RGB'],
            auc_rgb_ms=scores['RGB_MS'],delta_auc=scores['RGB_MS']-scores['RGB'])
        rows.append(result)
        print('AUC',fold,group,'RGB',round(scores['RGB'],3),
              'RGB+MS',round(scores['RGB_MS'],3),
              'delta',round(result['delta_auc'],3),flush=True)
RESULTS_MS=pd.DataFrame(rows)
RESULTS_MS.to_csv(MS_EXP/'04_rgb_vs_rgb_ms_spatial_auc.csv',index=False)
display(RESULTS_MS.round(3))
if RESULTS_MS.status.eq('evaluated').any():
    display(RESULTS_MS[RESULTS_MS.status.eq('evaluated')].groupby('acquisition').agg(
        folds=('fold','count'),mean_rgb=('auc_rgb','mean'),
        mean_rgb_ms=('auc_rgb_ms','mean'),mean_delta=('delta_auc','mean')).round(3))
else:
    print('No acquisition has enough held-out complete MS samples.'
          ' This is an important result; do not manufacture predictions.')
print('DONE:',MS_EXP)


## Scientific interpretation

A positive RGB+MS AUC difference is only **exploratory evidence** that sampled MS digital numbers carry additional information over *RGB pixel values* on the SAME matched support. It is **not** evidence of improvement over the pretrained RGB U-Net, nor evidence that full canopy-cover bias is corrected.

Because the original MS radiometry and image-to-image registration are not yet verified, do not interpret DN ratios as NDVI/NDRE or use this screening as definitive publication validation. The sampled nested inner roles are reused strictly. Sample size and spatial-unit counts matter more than a pixel-level p-value.

Only if there is encouraging held-out signal should we proceed to within-survey radiometric QA, local spectral contrasts, crown-level sampling, and eventual model fine-tuning. MOS 5–6 is **excluded completely**, awaiting source repair.
